<div style="
    margin: 20px auto;
    background: linear-gradient(145deg, #1c1917, #292524, #1c1917);
    border-radius: 24px;
    overflow: hidden;
    box-shadow: 0 20px 60px rgba(0,0,0,0.3);">
    <div style="padding: 50px 45px 20px;">
        <div style="display: inline-block; padding: 6px 16px; background: rgba(198,125,74,0.15); border-radius: 20px; margin-bottom: 20px;">
            <span style="color: #c67d4a; font-size: 12px; font-weight: 600; letter-spacing: 2px;">KAGGLE NLP COMPETITION</span>
        </div>
        <h1 style="color: #ffffff; font-size: 34px; font-weight: 800; margin: 0 0 15px; line-height: 1.15;">
           Natural Language Processing with Disaster Tweets<br>
            <span style="font-weight: 300; font-size: 24px; color: #a8a29e;">TF-IDF vs BiLSTM vs BERT</span>
        </h1>
        <p style="color: #a8a29e; font-size: 14.5px; line-height: 1.8; margin: 0 0 30px">
            Three generations of NLP compared on the same dataset. TF-IDF with keyword n-grams as a baseline, BiLSTM with GloVe embeddings and attention pooling for sequence awareness, and fine-tuned BERT for contextual understanding.
        </p>
    </div>
    <div style="display: flex; border-top: 1px solid rgba(255,255,255,0.06);">
        <div style="flex:1; padding: 18px 25px; border-right: 1px solid rgba(255,255,255,0.06);">
            <p style="color: #c67d4a; font-size: 22px; font-weight: 700; margin: 0;">10,876</p>
            <p style="color: #a8a29e; font-size: 11px; margin: 4px 0 0; text-transform: uppercase; letter-spacing: 1px;">Total Tweets</p>
        </div>
        <div style="flex:1; padding: 18px 25px; border-right: 1px solid rgba(255,255,255,0.06);">
            <p style="color: #c67d4a; font-size: 22px; font-weight: 700; margin: 0;">3</p>
            <p style="color: #a8a29e; font-size: 11px; margin: 4px 0 0; text-transform: uppercase; letter-spacing: 1px;">Model Approaches</p>
        </div>
        <div style="flex:1; padding: 18px 25px; border-right: 1px solid rgba(255,255,255,0.06);">
            <p style="color: #c67d4a; font-size: 22px; font-weight: 700; margin: 0;">F1</p>
            <p style="color: #a8a29e; font-size: 11px; margin: 4px 0 0; text-transform: uppercase; letter-spacing: 1px;">Eval Metric</p>
        </div>
        <div style="flex:1; padding: 18px 25px;">
            <p style="color: #c67d4a; font-size: 22px; font-weight: 700; margin: 0;">Binary</p>
            <p style="color: #a8a29e; font-size: 11px; margin: 4px 0 0; text-transform: uppercase; letter-spacing: 1px;">Classification</p>
        </div>
    </div>
</div>

<div style="
    margin: 0 auto 15px;
    padding: 18px 24px;
    background: #faf9f6;
    border-radius: 10px;
    border: 1px solid #e8e4de;">
    <p style="color: #c67d4a; font-size: 22px; font-weight: 600; margin: 0 0 12px;">Table of Contents</p>
    <p style="color: #44403c; font-size: 13.5px; line-height: 2.2; margin: 0;">
        <a href="#dataset-overview" style="color: #57534e; text-decoration: none;">01. Dataset Overview</a><br>
        <a href="#exploratory-analysis" style="color: #57534e; text-decoration: none;">02. Exploratory Data Analysis</a><br>
        <a href="#cleaning" style="color: #57534e; text-decoration: none;">03. Text Preprocessing</a><br>
        <a href="#tfidf" style="color: #57534e; text-decoration: none;">04. TF-IDF Baseline</a><br>
        <a href="#lstm" style="color: #57534e; text-decoration: none;">05. BiLSTM with GloVe and Attention</a><br>
        <a href="#bert" style="color: #57534e; text-decoration: none;">06. Fine-tuned BERT</a><br>
        <a href="#versus" style="color: #57534e; text-decoration: none;">07. Results</a><br>
        <a href="#submit" style="color: #57534e; text-decoration: none;">08. Submission</a><br>
        <a href="#takeaways" style="color: #57534e; text-decoration: none;">09. Observations</a><br>
    </p>
</div>

<a id="dataset-overview"></a>
<div style="
    margin: 35px auto 15px;
    padding: 12px 20px;
    background: #292524;
    border-radius: 8px;
    border-left: 3px solid #c67d4a;">
    <p style="color: #c67d4a; font-size: 10px; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin: 0 0 3px;">Section 01</p>
    <h2 style="color: #ffffff; font-size: 22px; font-weight: 600; margin: 0;">Dataset Overview</h2>
</div>

In [ ]:
import os
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import re, string, pickle

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
!pip install -q transformers==4.44.2 2>/dev/null

from collections import Counter

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
    recall_score, confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42); tf.random.set_seed(42)

# ── Palette ──
COPPER = '#c67d4a'
SLATE = '#457b9d'
SAND = '#d4a853'
CLAY = '#e07a5f'
SAGE = '#6b8f71'
CHARCOAL = '#292524'
MUTED = '#78716c'
PALETTE = ['#c67d4a', '#457b9d', '#d4a853', '#e07a5f', '#6b8f71',
           '#8b5e3c', '#5a7d9a', '#b8860b', '#c44536', '#7c9082']

plt.rcParams.update({
    'figure.facecolor': '#ffffff', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#e0e0e0', 'axes.labelcolor': '#444444',
    'axes.titlesize': 12, 'axes.labelsize': 10,
    'xtick.color': '#777777', 'ytick.color': '#777777',
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'legend.framealpha': 0.9, 'legend.edgecolor': '#e0e0e0',
    'grid.alpha': 0.12, 'grid.color': '#aaaaaa',
    'font.family': 'sans-serif', 'figure.dpi': 110,
})

train_df = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/train.csv')
test_df  = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/test.csv')
sub_df   = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/sample_submission.csv')

disaster    = train_df[train_df['target'] == 1]
no_disaster = train_df[train_df['target'] == 0]

(train_df.sample(8, random_state=42)[['text', 'keyword', 'target']]
 .style
 .set_properties(**{'text-align': 'left'})
 .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
)

<a id="exploratory-analysis"></a>
<div style="
    margin: 35px auto 15px;
    padding: 12px 20px;
    background: #292524;
    border-radius: 8px;
    border-left: 3px solid #c67d4a;">
    <p style="color: #c67d4a; font-size: 10px; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin: 0 0 3px;">Section 02</p>
    <h2 style="color: #ffffff; font-size: 22px; font-weight: 600; margin: 0;">Exploratory Data Analysis</h2>
</div>

In [ ]:
# ── Basic Stats ──
train_df['char_len'] = train_df['text'].str.len()
train_df['word_count'] = train_df['text'].str.split().str.len()

disaster    = train_df[train_df['target'] == 1]   # rebuild subsets so they include char_len and word_count
no_disaster = train_df[train_df['target'] == 0]

fig = plt.figure(figsize=(16, 20)) 
gs = gridspec.GridSpec(3, 2, wspace=0.35, hspace=0.45, height_ratios=[1, 1, 1])


# ── Class Balance ──
ax1 = fig.add_subplot(gs[0, 0])
counts = train_df['target'].value_counts().sort_index()
bars = ax1.bar(['Not Disaster', 'Disaster'], counts.values,
               color=[SLATE, COPPER], width=0.5, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{count:,} ({count/len(train_df)*100:.1f}%)',
             ha='center', fontsize=10, color='#444')
ax1.set_ylabel('Count')
ax1.set_title('Class balance', pad=10)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_ylim(0, counts.max() * 1.22) # 57-43 split, not terrible but enough to watch for recall issues on the minority class



# ── Char Length Boxplot by Class ──
ax5 = fig.add_subplot(gs[0, 1]) # boxplot shows the spread better than histograms — disaster tweets are slightly longer on average
bp = ax5.boxplot(
    [no_disaster['char_len'].values, disaster['char_len'].values],
    labels=['Not Disaster', 'Disaster'],
    patch_artist=True, widths=0.45,
    medianprops=dict(color='#333', linewidth=1.5),
    whiskerprops=dict(color='#999'),
    capprops=dict(color='#999'),
    flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#ccc')
)
for patch, color in zip(bp['boxes'], [SLATE, COPPER]):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

for i, subset in enumerate([no_disaster, disaster]):
    med = subset['char_len'].median()
    ax5.text(i + 1, med + 2, f'med={med:.0f}', ha='center', fontsize=9, color='#444') # printed medians on the plot, hard to read otherwise
ax5.set_ylabel('Characters')
ax5.set_title('Character length spread', pad=10)
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)
ax5.grid(axis='y')


# ── Character Length ──
ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(no_disaster['char_len'], bins=50, alpha=0.55, color=SLATE,
         label='Not Disaster', edgecolor='white', linewidth=0.3)
ax2.hist(disaster['char_len'], bins=50, alpha=0.55, color=COPPER,
         label='Disaster', edgecolor='white', linewidth=0.3)
ax2.axvline(140, color='#999', linestyle='--', linewidth=1, alpha=0.5) # old twitter limit — most tweets cluster below this
ax2.text(141, ax2.get_ylim()[1] * 0.02, '140', fontsize=8, color='#777', va='bottom')
ax2.set_xlabel('Characters')
ax2.set_ylabel('Frequency')
ax2.set_title('Tweet length distribution', pad=10)
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)


# ── Word Count ──
ax3 = fig.add_subplot(gs[1, 1])
ax3.hist(no_disaster['word_count'], bins=35, alpha=0.55,
         color=SLATE, label='Not Disaster', edgecolor='white', linewidth=0.3)
ax3.hist(disaster['word_count'], bins=35, alpha=0.55,
         color=COPPER, label='Disaster', edgecolor='white', linewidth=0.3)
ax3.set_xlabel('Words')
ax3.set_ylabel('Frequency')
ax3.set_title('Words per tweet', pad=10)
ax3.legend(fontsize=9)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)


# ── Missing Data ──
ax4 = fig.add_subplot(gs[2, 0])
cols = ['keyword', 'location', 'text']
present = [(train_df[c].notna().mean()) * 100 for c in cols] # keyword is present 99% of the time but location is only 33% — too sparse to rely on
missing = [100 - p for p in present]
bars_p = ax4.barh(cols, present, color=SAGE, height=0.5, edgecolor='white', label='Present')
bars_m = ax4.barh(cols, missing, left=present, color='#e0d5c8', height=0.5,
                  edgecolor='white', label='Missing')
for i, (p, m) in enumerate(zip(present, missing)):
    ax4.text(p/2, i, f'{p:.0f}%', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    if m > 5:
        ax4.text(p + m/2, i, f'{m:.0f}%', ha='center', va='center', fontsize=9, color='#888')
ax4.set_xlim(0, 100)
ax4.set_title('Data completeness', pad=10)
ax4.legend(fontsize=8, loc='lower right')
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)


# ── Duplicate Tweets ──
ax6 = fig.add_subplot(gs[2, 1]) 
dup_texts = train_df[train_df.duplicated(subset='text', keep=False)] 
dup_conflicting = dup_texts.groupby('text')['target'].nunique()
n_total_dups = train_df.duplicated(subset='text', keep='first').sum()
n_conflicting = (dup_conflicting > 1).sum()
n_consistent_dups = n_total_dups - n_conflicting
n_unique = len(train_df) - n_total_dups

categories = ['Unique tweets', 'Duplicate\n(consistent)', 'Duplicate\n(conflicting labels)']
values = [n_unique, n_consistent_dups, n_conflicting]
bar_colors = [SAGE, SAND, CLAY]

bars6 = ax6.barh(categories, values, color=bar_colors, height=0.5,
                 edgecolor='white', linewidth=1) # pie chart was messy for small slices, horizontal bar is cleaner
for bar, val in zip(bars6, values):
    pct = val / len(train_df) * 100
    ax6.text(bar.get_width() + len(train_df) * 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:,} ({pct:.1f}%)', va='center', fontsize=9, color='#444')
ax6.set_xlim(0, max(values) * 1.3)
ax6.set_title('Tweet uniqueness', pad=10)
ax6.spines['top'].set_visible(False)
ax6.spines['right'].set_visible(False)
ax6.spines['bottom'].set_visible(False)
ax6.set_xticks([]) # conflicting labels = same tweet text labeled both 0 and 1, noise the model has to absorb


plt.suptitle('Dataset Overview', fontsize=16, y=1.01, color='#333')
plt.tight_layout()
plt.show()

In [ ]:
# ── Keyword Analysis ──
keyword_stats = (train_df.dropna(subset=['keyword'])
                 .groupby('keyword')['target']
                 .agg(['mean', 'count'])
                 .rename(columns={'mean': 'disaster_rate', 'count': 'total'})
                 .query('total >= 10')  # filter out rare keywords, too noisy to draw conclusions
                 .sort_values('disaster_rate'))

fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(2, 2, wspace=0.35, hspace=0.4, height_ratios=[1.6, 1])


# ── Top Disaster Keywords ──
ax1 = fig.add_subplot(gs[0, 0])
top = keyword_stats.tail(15)
bars_top = ax1.barh(top.index, top['disaster_rate'],
                    color=[COPPER if r > 0.8 else SAND for r in top['disaster_rate']],
                    height=0.7, edgecolor='white', linewidth=0.5)
for i, (rate, n) in enumerate(zip(top['disaster_rate'], top['total'])):
    ax1.text(rate + 0.01, i, f'{rate:.0%} (n={n})', va='center', fontsize=9, color='#444')
ax1.set_xlim(0, 1.15)
ax1.set_xlabel('Disaster rate')
ax1.set_title('Keywords most tied to real disasters', pad=10)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)


# ── Bottom Disaster Keywords ──
ax2 = fig.add_subplot(gs[0, 1])
bottom = keyword_stats.head(15)
bars_bot = ax2.barh(bottom.index, bottom['disaster_rate'],
                    color=[SLATE if r < 0.15 else MUTED for r in bottom['disaster_rate']],
                    height=0.7, edgecolor='white', linewidth=0.5)
for i, (rate, n) in enumerate(zip(bottom['disaster_rate'], bottom['total'])):
    ax2.text(rate + 0.01, i, f'{rate:.0%} (n={n})', va='center', fontsize=9, color='#444')
ax2.set_xlim(0, 0.5)
ax2.set_xlabel('Disaster rate')
ax2.set_title('Keywords almost never literal', pad=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)


# ── Keyword Signal Buckets ──
ax3 = fig.add_subplot(gs[1, 0]) # cleaner than a histogram — just show how many keywords fall into each signal tier
buckets = {
    'Strong disaster\n(≥80%)': (keyword_stats['disaster_rate'] >= 0.8).sum(),
    'Leaning disaster\n(60-80%)': keyword_stats['disaster_rate'].between(0.6, 0.8, inclusive='left').sum(),
    'Ambiguous\n(40-60%)': keyword_stats['disaster_rate'].between(0.4, 0.6, inclusive='left').sum(),
    'Leaning safe\n(20-40%)': keyword_stats['disaster_rate'].between(0.2, 0.4, inclusive='left').sum(),
    'Strong safe\n(≤20%)': (keyword_stats['disaster_rate'] < 0.2).sum(),
}
bucket_colors = [COPPER, SAND, MUTED, '#a8b5c0', SLATE]
bars3 = ax3.bar(range(len(buckets)), list(buckets.values()),
                color=bucket_colors, width=0.6, edgecolor='white', linewidth=1)
ax3.set_xticks(range(len(buckets)))
ax3.set_xticklabels(list(buckets.keys()), fontsize=9)
for bar, val in zip(bars3, buckets.values()):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(val), ha='center', fontsize=10, color='#333')
ax3.set_ylabel('Number of keywords')
ax3.set_title('Keyword signal strength tiers', pad=10)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False) # most keywords land at the extremes so keyword alone is a decent feature


# ── Keyword Coverage ──
ax4 = fig.add_subplot(gs[1, 1]) # how much of the dataset do high-signal keywords actually cover?
strong_disaster_kw = keyword_stats[keyword_stats['disaster_rate'] >= 0.8].index
strong_safe_kw = keyword_stats[keyword_stats['disaster_rate'] < 0.2].index
ambiguous_kw = keyword_stats[keyword_stats['disaster_rate'].between(0.3, 0.7)].index

kw_present = train_df.dropna(subset=['keyword'])
n_strong_d = kw_present[kw_present['keyword'].isin(strong_disaster_kw)].shape[0]
n_strong_s = kw_present[kw_present['keyword'].isin(strong_safe_kw)].shape[0]
n_ambig = kw_present[kw_present['keyword'].isin(ambiguous_kw)].shape[0]
n_other = len(kw_present) - n_strong_d - n_strong_s - n_ambig

coverage = [n_strong_d, n_strong_s, n_ambig, n_other]
coverage_labels = ['Strong disaster\nkeywords', 'Strong safe\nkeywords',
                   'Ambiguous\nkeywords', 'Other / rare\nkeywords']
coverage_colors = [COPPER, SLATE, MUTED, '#e0d5c8']

bars4 = ax4.bar(range(len(coverage)), coverage,
                color=coverage_colors, width=0.6, edgecolor='white', linewidth=1)
for bar, val in zip(bars4, coverage):
    pct = val / len(kw_present) * 100
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{val:,} ({pct:.0f}%)', ha='center', fontsize=9, color='#333')
ax4.set_xticks(range(len(coverage)))
ax4.set_xticklabels(coverage_labels, fontsize=9)
ax4.set_ylabel('Number of tweets')
ax4.set_title('Tweet coverage by keyword signal tier', pad=10)
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False) # decent chunk can be classified by keyword alone, rest needs the model

plt.suptitle('Keyword Signal Analysis', fontsize=16, y=1.01, color='#333')
plt.tight_layout()
plt.show()

In [ ]:
# ── Vocabulary Analysis ──
def word_freq(texts): # score each word by how much it leans toward one class vs the other
    words = ' '.join(texts).lower().split()
    return Counter(w.strip(string.punctuation) for w in words if len(w.strip(string.punctuation)) > 2)

d_freq = word_freq(disaster['text'])
s_freq = word_freq(no_disaster['text'])
d_total, s_total = sum(d_freq.values()), sum(s_freq.values())

scores = {} # +1 means only in disaster, -1 means only in non-disaster
for w in set(list(d_freq.keys())[:2000]) | set(list(s_freq.keys())[:2000]):
    if d_freq.get(w, 0) + s_freq.get(w, 0) >= 15:  # min frequency to avoid noise
        dr = d_freq.get(w, 0) / d_total
        sr = s_freq.get(w, 0) / s_total
        scores[w] = (dr - sr) / (dr + sr + 1e-9)

ranked = sorted(scores.items(), key=lambda x: x[1])

fig = plt.figure(figsize=(18, 16)) # went with relative frequency difference because chi-squared and PMI were harder to interpret
gs = gridspec.GridSpec(2, 2, wspace=0.35, hspace=0.4, height_ratios=[1.6, 1])


# ── Disaster Vocabulary ──
ax1 = fig.add_subplot(gs[0, 0])
top_d = ranked[-15:][::-1]
words_d, vals_d = zip(*top_d)
bars1 = ax1.barh(range(len(words_d)), vals_d,
                 color=[COPPER if v > 0.5 else SAND for v in vals_d],
                 height=0.7, edgecolor='white', linewidth=0.5)
ax1.set_yticks(range(len(words_d)))
ax1.set_yticklabels(words_d, fontsize=9)
for i, v in enumerate(vals_d):
    ax1.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=8, color='#444')
ax1.set_xlabel('Relative frequency score')
ax1.set_title('Vocabulary leaning disaster', pad=10)
ax1.invert_yaxis()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)


# ── Non-Disaster Vocabulary ──
ax2 = fig.add_subplot(gs[0, 1])
top_s = ranked[:15]
words_s, vals_s = zip(*top_s)
vals_abs = [abs(v) for v in vals_s]
bars2 = ax2.barh(range(len(words_s)), vals_abs,
                 color=[SLATE if v > 0.5 else MUTED for v in vals_abs],
                 height=0.7, edgecolor='white', linewidth=0.5)
ax2.set_yticks(range(len(words_s)))
ax2.set_yticklabels(words_s, fontsize=9)
for i, v in enumerate(vals_abs):
    ax2.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=8, color='#444')
ax2.set_xlabel('Relative frequency score')
ax2.set_title('Vocabulary leaning not disaster', pad=10)
ax2.invert_yaxis()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)


# ── Shared Vocabulary Overlap ──
ax3 = fig.add_subplot(gs[1, 0]) # how many words appear frequently in both classes? these are the ambiguous ones models struggle with
d_top_words = set(w for w, _ in d_freq.most_common(500))
s_top_words = set(w for w, _ in s_freq.most_common(500))
only_d = len(d_top_words - s_top_words)
only_s = len(s_top_words - d_top_words)
shared = len(d_top_words & s_top_words)

overlap_cats = ['Disaster only', 'Shared', 'Non-disaster only']
overlap_vals = [only_d, shared, only_s]
overlap_colors = [COPPER, MUTED, SLATE]

bars3 = ax3.bar(overlap_cats, overlap_vals, color=overlap_colors,
                width=0.5, edgecolor='white', linewidth=1)
for bar, val in zip(bars3, overlap_vals):
    pct = val / 500 * 100
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             f'{val} ({pct:.0f}%)', ha='center', fontsize=9, color='#333')
ax3.set_ylabel('Words (top 500 per class)')
ax3.set_title('Vocabulary overlap between classes', pad=10)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False) 


# ── Most Ambiguous Words ──
ax4 = fig.add_subplot(gs[1, 1]) # words with high frequency in both classes but near-zero score — these fail TF-IDF
ambiguous_words = sorted(scores.items(), key=lambda x: abs(x[1]))[:15]
amb_words, amb_vals = zip(*ambiguous_words)
amb_freq = [d_freq.get(w, 0) + s_freq.get(w, 0) for w in amb_words]

sorted_idx = np.argsort(amb_freq)[::-1] # sort by total frequency so the most common ambiguous words are on top
amb_words = [amb_words[i] for i in sorted_idx]
amb_vals = [amb_vals[i] for i in sorted_idx]
amb_freq = [amb_freq[i] for i in sorted_idx]

bars4 = ax4.barh(range(len(amb_words)), amb_freq,
                 color=MUTED, height=0.6, edgecolor='white', linewidth=0.5, alpha=0.7)
ax4.set_yticks(range(len(amb_words)))
ax4.set_yticklabels(amb_words, fontsize=9)
for i, (freq, score) in enumerate(zip(amb_freq, amb_vals)):
    ax4.text(freq + max(amb_freq) * 0.02, i,
             f'n={freq}  score={score:+.3f}', va='center', fontsize=8, color='#444')
ax4.set_xlabel('Total frequency (both classes)')
ax4.set_title('Most ambiguous words (near-zero class signal)', pad=10)
ax4.invert_yaxis()
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False) # "like", "people", "new" etc. carry zero signal, this is where BERT should help

plt.suptitle('Vocabulary Analysis', fontsize=16, y=1.01, color='#333')
plt.tight_layout()
plt.show()

<a id="cleaning"></a>
<div style="
    margin: 35px auto 15px;
    padding: 12px 20px;
    background: #292524;
    border-radius: 8px;
    border-left: 3px solid #c67d4a;">
    <p style="color: #c67d4a; font-size: 10px; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin: 0 0 3px;">Section 03</p>
    <h2 style="color: #ffffff; font-size: 22px; font-weight: 600; margin: 0;">Text Preprocessing</h2>
</div>


<div style="
    margin: 0 auto 15px;
    padding: 18px 24px;
    background: #faf9f6;
    border-radius: 10px;
    border: 1px solid #e8e4de;">
    <p style="color: #44403c; font-size: 14px; line-height: 1.8; margin: 0;">
        Two pipelines. The heavy one strips URLs, mentions, hashtags, punctuation and lowercases everything. This is for TF-IDF and LSTM where clean tokens matter. The light one replaces URLs and mentions with placeholder tokens but keeps casing and structure intact. BERT's tokenizer handles subword splitting internally and benefits from seeing the original text shape.
    </p>
</div>


In [ ]:
def clean_heavy(text):
    """Strip noise for TF-IDF and LSTM."""
    text = re.sub(r'http\S+|www\.\S+', ' url ', text)   # was deleting entirely — keeping token helps TF-IDF
    text = re.sub(r'@\w+', ' user ', text)               # mentions carry structural signal
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()
    # tried lemmatization with spacy — slower and no F1 improvement
    # tried removing stopwords — hurt performance, words like "not" matter here

def clean_light(text, keyword=''):
    """Minimal cleaning for BERT."""
    text = re.sub(r'http\S+|www\.\S+', '[URL]', text)
    text = re.sub(r'@\w+', '[USER]', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if keyword:
        text = f"{keyword} : {text}" # prepend keyword, BERT's attention can learn to use it as context
    return text


# ── Keyword Cleaning ──
train_df['keyword_clean'] = train_df['keyword'].fillna('').str.replace('%20', ' ', regex=False).str.lower() # %20 encoded spaces in keyword column
test_df['keyword_clean'] = test_df['keyword'].fillna('').str.replace('%20', ' ', regex=False).str.lower()

train_df['text_clean'] = train_df['text'].apply(clean_heavy)
test_df['text_clean']  = test_df['text'].apply(clean_heavy)

train_df['text_bert'] = train_df.apply(
    lambda r: clean_light(r['text'], r['keyword_clean']), axis=1) # EDA showed keyword alone classifies ~40% of tweets correctly
test_df['text_bert'] = test_df.apply(
    lambda r: clean_light(r['text'], r['keyword_clean']), axis=1)


# ── Keyword-Enhanced Text for TF-IDF ──
train_df['text_kw'] = train_df['text_clean'] + ' ' + train_df['keyword_clean'] # tried as a separate feature column but concatenation worked better with n-grams
test_df['text_kw'] = test_df['text_clean'] + ' ' + test_df['keyword_clean']

# show side by side
samples = train_df.sample(4, random_state=7)
for _, row in samples.iterrows():
    print(f"  raw   : {row['text'][:90]}")
    print(f"  heavy : {row['text_clean'][:90]}")
    print(f"  kw    : {row['text_kw'][:90]}")
    print(f"  bert  : {row['text_bert'][:90]}")
    print()

X_text = train_df['text_clean'].values
X_text_kw = train_df['text_kw'].values
X_bert = train_df['text_bert'].values
y = train_df['target'].values

splits = train_test_split(X_text, X_bert, X_text_kw, y, test_size=0.15, random_state=42, stratify=y) # split all text arrays together so indices stay aligned
X_train_text, X_val_text = splits[0], splits[1]
X_train_bert, X_val_bert = splits[2], splits[3]
X_train_kw, X_val_kw = splits[4], splits[5]
y_train, y_val = splits[6], splits[7]

print(f"  train: {len(y_train):,}  |  val: {len(y_val):,}  |  test: {len(test_df):,}")
print(f"  train disaster rate: {y_train.mean():.1%}  |  val: {y_val.mean():.1%}")

<a id="tfidf"></a>
<div style="
    margin: 35px auto 15px;
    padding: 12px 20px;
    background: #292524;
    border-radius: 8px;
    border-left: 3px solid #c67d4a;">
    <p style="color: #c67d4a; font-size: 10px; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin: 0 0 3px;">Section 04</p>
    <h2 style="color: #ffffff; font-size: 22px; font-weight: 600; margin: 0;">TF-IDF Baseline</h2>
</div>

<div style="
    margin: 0 auto 15px;
    padding: 18px 24px;
    background: #faf9f6;
    border-radius: 10px;
    border: 1px solid #e8e4de;">
    <p style="color: #44403c; font-size: 14px; line-height: 1.8; margin: 0;">
        TF-IDF weights each word by how often it appears in a tweet versus how common it is across all tweets. A word like "earthquake" that shows up in a few tweets gets a high weight. A word like "the" that appears everywhere gets almost zero. The result is a sparse vector per tweet where dimensions correspond to vocabulary terms. No word order, no context, just weighted presence. Three classifiers run on top: logistic regression, naive bayes and a linear SVM. A soft voting ensemble combines all three.
    </p>
</div>


In [ ]:
tfidf = TfidfVectorizer(
    max_features=25000,
    ngram_range=(1, 3),
    min_df=2, max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
)

X_train_tfidf = tfidf.fit_transform(X_train_kw)
X_val_tfidf   = tfidf.transform(X_val_kw)
X_test_tfidf  = tfidf.transform(test_df['text_kw'].values)


print(f"  vocabulary: {len(tfidf.vocabulary_):,} terms")
print(f"  matrix: {X_train_tfidf.shape}")

models_tfidf = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Multinomial NB': MultinomialNB(alpha=0.1),
    'Linear SVM': CalibratedClassifierCV(
        LinearSVC(C=0.5, max_iter=2000, random_state=42), cv=3
    ),
}

tfidf_results = {}

print(f"\n  {'MODEL':<25s}  {'F1':>6s}  {'ACC':>6s}  {'PREC':>6s}  {'REC':>6s}")
print(f"  {'─'*55}")

for name, model in models_tfidf.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_val_tfidf)
    probs = model.predict_proba(X_val_tfidf)[:, 1]

    res = {
        'model': model, 'preds': preds, 'probs': probs,
        'f1': f1_score(y_val, preds),
        'accuracy': accuracy_score(y_val, preds),
        'precision': precision_score(y_val, preds),
        'recall': recall_score(y_val, preds),
    }
    tfidf_results[name] = res
    print(f"  {name:<25s}  {res['f1']:>.4f}  {res['accuracy']:>.4f}  {res['precision']:>.4f}  {res['recall']:>.4f}")

# ensemble
ensemble = VotingClassifier(
    estimators=[(k.lower().replace(' ', '_'), v) for k, v in models_tfidf.items()],
    voting='soft'
)
ensemble.fit(X_train_tfidf, y_train)
ens_preds = ensemble.predict(X_val_tfidf)
ens_probs = ensemble.predict_proba(X_val_tfidf)[:, 1]

tfidf_results['Ensemble'] = {
    'model': ensemble, 'preds': ens_preds, 'probs': ens_probs,
    'f1': f1_score(y_val, ens_preds),
    'accuracy': accuracy_score(y_val, ens_preds),
    'precision': precision_score(y_val, ens_preds),
    'recall': recall_score(y_val, ens_preds),
}
print(f"\n  {'Ensemble':<25s}  {tfidf_results['Ensemble']['f1']:>.4f}  {tfidf_results['Ensemble']['accuracy']:>.4f}  {tfidf_results['Ensemble']['precision']:>.4f}  {tfidf_results['Ensemble']['recall']:>.4f}")

best_name = max(tfidf_results, key=lambda k: tfidf_results[k]['f1'])
cv_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=25000, ngram_range=(1,3), min_df=2, max_df=0.95, sublinear_tf=True, strip_accents='unicode')),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
]) # cross-validation on best, used LogisticRegression to avoid nested CV issues with CalibratedClassifierCV

cv = cross_val_score(cv_pipe, X_text_kw, y, cv=5, scoring='f1')
print(f"\n  best: {best_name}  |  5-fold CV F1 (LogReg): {cv.mean():.4f} ± {cv.std():.4f}")

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
theme_cmap = LinearSegmentedColormap.from_list('theme', ['#ffffff', '#e8d5c4', COPPER], N=256)

fig, axes = plt.subplots(2, 2, figsize=(12, 11))

for ax, (name, res) in zip(axes.flat, tfidf_results.items()):
    cm = confusion_matrix(y_val, res['preds'])
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap=theme_cmap,
                vmin=0, vmax=1,
                xticklabels=['Not', 'Disaster'], yticklabels=['Not', 'Disaster'],
                ax=ax, cbar=False, linewidths=1.5, linecolor='white',
                annot_kws={'size': 12, 'fontweight': 'bold'})
    ax.set_title(f'{name}\nF1={res["f1"]:.3f}', fontsize=10, pad=8)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True', fontsize=11)

plt.suptitle('TF-IDF Classifiers (Normalized Confusion Matrices)', fontsize=16, y=1.02, color='#333')
plt.tight_layout()
plt.show()

<a id="lstm"></a>
<div style="
    margin: 35px auto 15px;
    padding: 12px 20px;
    background: #292524;
    border-radius: 8px;
    border-left: 3px solid #c67d4a;">
    <p style="color: #c67d4a; font-size: 10px; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin: 0 0 3px;">Section 05</p>
    <h2 style="color: #ffffff; font-size: 22px; font-weight: 600; margin: 0;">BiLSTM with GloVe and Attention</h2>
</div>

<div style="
    margin: 0 auto 15px;
    padding: 18px 24px;
    background: #faf9f6;
    border-radius: 10px;
    border: 1px solid #e8e4de;">
    <p style="color: #44403c; font-size: 14px; line-height: 1.8; margin: 0;">
        TF-IDF throws away word order entirely. "Man bites dog" and "dog bites man" produce the same vector. An LSTM reads tokens sequentially and maintains a hidden state that accumulates meaning as it moves through the sentence. Bidirectional means two passes, one forward and one backward so every position has context from both sides. GloVe embeddings trained on 2 billion tweets initialize the word vectors so the model starts with real semantic relationships instead of random weights.
    </p>
</div>


In [ ]:
MAX_VOCAB = 20000
MAX_LEN = 60
EMBED_DIM = 100

tokenizer_lstm = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer_lstm.fit_on_texts(X_train_text)

X_train_seq = pad_sequences(tokenizer_lstm.texts_to_sequences(X_train_text), maxlen=MAX_LEN)
X_val_seq   = pad_sequences(tokenizer_lstm.texts_to_sequences(X_val_text), maxlen=MAX_LEN)
X_test_seq  = pad_sequences(tokenizer_lstm.texts_to_sequences(test_df['text_clean'].values), maxlen=MAX_LEN)

word_index = tokenizer_lstm.word_index
vocab_size = min(MAX_VOCAB, len(word_index)) + 1
print(f"  vocabulary: {vocab_size:,} tokens  |  sequence length: {MAX_LEN}")

# ── GloVe ──
GLOVE_PATH = '/kaggle/input/datasets/robertyoung/glove-twitter-pickles-27b-25d-50d-100d-200d/glove.twitter.27B.100d.pkl'

with open(GLOVE_PATH, 'rb') as f:
    embeddings_index = pickle.load(f)

embedding_matrix = np.zeros((vocab_size, EMBED_DIM))
found = 0
for word, idx in word_index.items():
    if idx >= vocab_size:
        continue
    vec = embeddings_index.get(word)
    if vec is not None:
        embedding_matrix[idx] = vec
        found += 1

print(f"  GloVe vectors: {len(embeddings_index):,}")
print(f"  coverage: {found:,}/{vocab_size-1:,} ({found/(vocab_size-1)*100:.1f}%)")

def build_bilstm():
    inp = keras.Input(shape=(MAX_LEN,))

    x = layers.Embedding(vocab_size, EMBED_DIM, weights=[embedding_matrix],
                         trainable=False)(inp) # frozen GloVe, tried trainable=True from the start but it overfit faster on 6k samples
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True,
                             dropout=0.2, recurrent_dropout=0.2))(x)  # recurrent_dropout disables cuDNN but gives better regularization on small data

    # tried plain GlobalMaxPool but attention gave ~1% F1 boost on this dataset
    attention = layers.Dense(1, activation='tanh')(x)
    attention = layers.Flatten()(attention)
    attention = layers.Activation('softmax')(attention)
    attention = layers.RepeatVector(128)(attention)  # 128 = 64*2 bidirectional
    attention = layers.Permute([2, 1])(attention)
    x = layers.Multiply()([x, attention])
    x = layers.Lambda(lambda z: tf.keras.backend.sum(z, axis=1))(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inp, out, name='BiLSTM_Attention')

lstm_model = build_bilstm()
lstm_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

trainable = sum(tf.keras.backend.count_params(w) for w in lstm_model.trainable_weights)
print(f"  {lstm_model.name}  |  params: {lstm_model.count_params():,}  |  trainable: {trainable:,}")

history_lstm = lstm_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=30, batch_size=64,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                    patience=2, min_lr=1e-6, verbose=1),
    ],
    verbose=2
)

# ── threshold optimization for LSTM too ──
lstm_probs = lstm_model.predict(X_val_seq, verbose=0).flatten()

best_lstm_thresh = 0.5
best_lstm_f1 = 0
for t in np.arange(0.30, 0.70, 0.01):
    preds_t = (lstm_probs >= t).astype(int)
    f1_t = f1_score(y_val, preds_t)
    if f1_t > best_lstm_f1:
        best_lstm_f1 = f1_t
        best_lstm_thresh = t

lstm_preds = (lstm_probs >= best_lstm_thresh).astype(int)
lstm_f1 = f1_score(y_val, lstm_preds)
print(f"\n  BiLSTM val F1={lstm_f1:.4f}  Acc={accuracy_score(y_val, lstm_preds):.4f}")
print(f"  optimal threshold: {best_lstm_thresh:.2f} (default 0.50 would give F1={f1_score(y_val, (lstm_probs >= 0.5).astype(int)):.4f})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
epochs = range(1, len(history_lstm.history['accuracy']) + 1)

ax1.plot(epochs, history_lstm.history['accuracy'], color=COPPER, linewidth=1.8, label='Train')
ax1.plot(epochs, history_lstm.history['val_accuracy'], color=SLATE, linewidth=1.8, linestyle='--', label='Val')
ax1.fill_between(epochs, history_lstm.history['accuracy'],
                 history_lstm.history['val_accuracy'], alpha=0.06, color=COPPER)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.set_title('BiLSTM — Accuracy', pad=8)
ax1.legend(); ax1.grid(True)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

ax2.plot(epochs, history_lstm.history['loss'], color=COPPER, linewidth=1.8, label='Train')
ax2.plot(epochs, history_lstm.history['val_loss'], color=SLATE, linewidth=1.8, linestyle='--', label='Val')
ax2.fill_between(epochs, history_lstm.history['loss'],
                 history_lstm.history['val_loss'], alpha=0.06, color=SLATE)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.set_title('BiLSTM — Loss', pad=8)
ax2.legend(); ax2.grid(True)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

<a id="bert"></a>
<div style="
    margin: 35px auto 15px;
    padding: 12px 20px;
    background: #292524;
    border-radius: 8px;
    border-left: 3px solid #c67d4a;">
    <p style="color: #c67d4a; font-size: 10px; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin: 0 0 3px;">Section 06</p>
    <h2 style="color: #ffffff; font-size: 22px; font-weight: 600; margin: 0;">Fine-tuned BERT</h2>
</div>

<div style="
    margin: 0 auto 15px;
    padding: 18px 24px;
    background: #faf9f6;
    border-radius: 10px;
    border: 1px solid #e8e4de;">
    <p style="color: #44403c; font-size: 14px; line-height: 1.8; margin: 0;">
    GloVe gives every word one fixed vector regardless of context. "Fire" produces the same embedding whether the tweet reports a building burning or someone getting fired from a job. BERT solves this with self-attention where each token's representation is computed from every other token in the sentence simultaneously. The word "fire" in "the building is on fire" attends to "building" and produces a very different embedding than "fire" in "he got fired last week" which attends to "got". The model was pre-trained on billions of words using masked language modeling so it already understands English deeply. Fine-tuning just teaches it what disaster versus not-disaster looks like on top of that existing knowledge.
</p>
</div>


In [ ]:
BERT_MODEL = 'bert-base-uncased'
BERT_MAX_LEN = 64
BERT_BATCH = 16
BERT_EPOCHS = 6     
BERT_LR = 1e-5 # 2e-5 was overfitting by epoch 3, halved it

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

def tokenize_bert(texts, max_len=BERT_MAX_LEN):
    return bert_tokenizer(
        list(texts), padding='max_length', truncation=True,
        max_length=max_len, return_tensors='tf'
    )

train_enc = tokenize_bert(X_train_bert)
val_enc   = tokenize_bert(X_val_bert)
test_enc  = tokenize_bert(test_df['text_bert'].values)

print(f"  model: {BERT_MODEL}")
print(f"  max length: {BERT_MAX_LEN}  |  batch: {BERT_BATCH}  |  lr: {BERT_LR}")

from tf_keras.optimizers import Adam as TFKerasAdam # Keras 3 and tf-keras optimizers are incompatible, HuggingFace TF models use tf-keras internally

optimizer = TFKerasAdam(
    learning_rate=BERT_LR,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-8,
    weight_decay=0.01, # tried 0.1 weight decay but 0.01 was enough here
)

bert_model = TFAutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=2)
bert_model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print(f"  parameters: {bert_model.count_params():,}")

history_bert = bert_model.fit(
    dict(train_enc), y_train,
    validation_data=(dict(val_enc), y_val),
    epochs=BERT_EPOCHS, batch_size=BERT_BATCH,
    callbacks=[
        callbacks.EarlyStopping(
            monitor='val_loss', patience=2, # patience=2 because BERT loss can fluctuate between epochs on 6k samples
            restore_best_weights=True, verbose=1
        )
    ],
    verbose=2
)

# ── Threshold Optimization ──
bert_logits = bert_model.predict(dict(val_enc), verbose=0).logits
bert_probs  = tf.nn.softmax(bert_logits, axis=1).numpy()[:, 1]

best_thresh = 0.5
best_f1 = 0
for t in np.arange(0.30, 0.70, 0.01):
    preds_t = (bert_probs >= t).astype(int)
    f1_t = f1_score(y_val, preds_t)
    if f1_t > best_f1:
        best_f1 = f1_t
        best_thresh = t

bert_preds = (bert_probs >= best_thresh).astype(int)
bert_f1 = f1_score(y_val, bert_preds)

print(f"\n  BERT val F1={bert_f1:.4f}  Acc={accuracy_score(y_val, bert_preds):.4f}")
print(f"  optimal threshold: {best_thresh:.2f} (default 0.50 would give F1={f1_score(y_val, (bert_probs >= 0.5).astype(int)):.4f})")

<a id="versus"></a>
<div style="
    margin: 35px auto 15px;
    padding: 12px 20px;
    background: #292524;
    border-radius: 8px;
    border-left: 3px solid #c67d4a;">
    <p style="color: #c67d4a; font-size: 10px; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin: 0 0 3px;">Section 07</p>
    <h2 style="color: #ffffff; font-size: 22px; font-weight: 600; margin: 0;">Results</h2>
</div>


In [ ]:
best_tfidf_name = max(tfidf_results, key=lambda k: tfidf_results[k]['f1'])
best_tfidf = tfidf_results[best_tfidf_name]

all_models = {
    f'TF-IDF ({best_tfidf_name})': best_tfidf,
    'BiLSTM + GloVe': {
        'preds': lstm_preds, 'probs': lstm_probs,
        'f1': lstm_f1, 'accuracy': accuracy_score(y_val, lstm_preds),
        'precision': precision_score(y_val, lstm_preds),
        'recall': recall_score(y_val, lstm_preds),
    },
    'BERT': {
        'preds': bert_preds, 'probs': bert_probs,
        'f1': bert_f1, 'accuracy': accuracy_score(y_val, bert_preds),
        'precision': precision_score(y_val, bert_preds),
        'recall': recall_score(y_val, bert_preds),
    },
}

print(f"  {'MODEL':<28s}  {'F1':>6s}  {'ACC':>6s}  {'PREC':>6s}  {'REC':>6s}")
print(f"  {'─'*58}")
for name, res in all_models.items():
    print(f"  {name:<28s}  {res['f1']:>.4f}  {res['accuracy']:>.4f}  {res['precision']:>.4f}  {res['recall']:>.4f}")

# ── Bar Chart ──
fig, ax = plt.subplots(figsize=(12, 5))
names = list(all_models.keys())
metrics = ['f1', 'precision', 'recall']
labels = ['F1', 'Precision', 'Recall']
x = np.arange(len(names))
w = 0.22
colors = [COPPER, SLATE, SAND]

for i, (m, lab, col) in enumerate(zip(metrics, labels, colors)):
    vals = [all_models[n][m] for n in names]
    bars = ax.bar(x + i*w, vals, w, label=lab, color=col, edgecolor='white', linewidth=1, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', fontsize=8, color='#444')

ax.set_xticks(x + w)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylim(0.55, 1.0)
ax.set_ylabel('Score')
ax.set_title('Model Comparison', fontsize=12, pad=10)
ax.legend(loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
curve_colors = [COPPER, SLATE, SAND]

for i, (name, res) in enumerate(all_models.items()):
    fpr, tpr, _ = roc_curve(y_val, res['probs'])
    ax1.plot(fpr, tpr, color=curve_colors[i], linewidth=2,
             label=f'{name} (AUC={auc(fpr, tpr):.3f})')

    prec_c, rec_c, _ = precision_recall_curve(y_val, res['probs'])
    ap = average_precision_score(y_val, res['probs'])
    ax2.plot(rec_c, prec_c, color=curve_colors[i], linewidth=2,
             label=f'{name} (AP={ap:.3f})')

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1)
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves', pad=10)
ax1.legend(fontsize=9, loc='lower right')
ax1.grid(True)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

ax2.axhline(y_val.mean(), color='#999', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves', pad=10)
ax2.legend(fontsize=9, loc='lower left')
ax2.grid(True)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
error_df = pd.DataFrame({
    'text': X_val_text,
    'true': y_val,
    'tfidf': best_tfidf['preds'],
    'lstm': lstm_preds,
    'bert': bert_preds,
})

all_wrong = error_df[
    (error_df['tfidf'] != error_df['true']) &
    (error_df['lstm'] != error_df['true']) &
    (error_df['bert'] != error_df['true'])
]

bert_wins = error_df[
    (error_df['bert'] == error_df['true']) &
    ((error_df['tfidf'] != error_df['true']) | (error_df['lstm'] != error_df['true']))
]

tfidf_only = error_df[
    (error_df['tfidf'] == error_df['true']) &
    (error_df['bert'] != error_df['true'])
]

print(f"  all three wrong:          {len(all_wrong)}")
print(f"  BERT right, others wrong: {len(bert_wins)}")
print(f"  TF-IDF right, BERT wrong: {len(tfidf_only)}")

print(f"\n  tweets every model got wrong:")
print(f"  {'─'*65}")
for _, row in all_wrong.head(6).iterrows():
    tag = 'DISASTER' if row['true'] == 1 else 'NOT'
    print(f"  [{tag}] {row['text'][:90]}")
    print()

<a id="submit"></a>
<div style="
    margin: 35px auto 15px;
    padding: 12px 20px;
    background: #292524;
    border-radius: 8px;
    border-left: 3px solid #c67d4a;">
    <p style="color: #c67d4a; font-size: 10px; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin: 0 0 3px;">Section 08</p>
    <h2 style="color: #ffffff; font-size: 22px; font-weight: 600; margin: 0;">Submission</h2>
</div>


In [ ]:
best_name = max(all_models, key=lambda k: all_models[k]['f1'])
print(f"  best model: {best_name} (F1={all_models[best_name]['f1']:.4f})")

if 'BERT' in best_name:
    test_logits = bert_model.predict(dict(test_enc), verbose=0).logits
    test_probs = tf.nn.softmax(test_logits, axis=1).numpy()[:, 1]
    # use the optimized threshold from validation, not default 0.5
    test_preds = (test_probs >= best_thresh).astype(int)
    print(f"  using optimized threshold: {best_thresh:.2f}")
elif 'LSTM' in best_name:
    test_preds = (lstm_model.predict(X_test_seq, verbose=0).flatten() >= 0.5).astype(int)
else:
    test_preds = best_tfidf['model'].predict(X_test_tfidf)

sub_df['target'] = test_preds
sub_df.to_csv('submission.csv', index=False)

print(f"  predictions: {len(test_preds):,}")
print(f"  disaster: {(test_preds == 1).sum():,} ({test_preds.mean():.1%})")
print(f"  not disaster: {(test_preds == 0).sum():,} ({(1 - test_preds).mean():.1%})")
print(f"  saved to submission.csv")

sub_df.head(10)

<a id="takeaways"></a>
<div style="
    margin: 35px auto 15px;
    padding: 12px 20px;
    background: #292524;
    border-radius: 8px;
    border-left: 3px solid #c67d4a;">
    <p style="color: #c67d4a; font-size: 10px; font-weight: 600; letter-spacing: 2px; text-transform: uppercase; margin: 0 0 3px;">Section 09</p>
    <h2 style="color: #ffffff; font-size: 22px; font-weight: 600; margin: 0;">Observations</h2>
</div>
<div style="
    margin: 0 auto 12px;
    padding: 18px 24px;
    background: #faf9f6;
    border-radius: 10px;
    border: 1px solid #e8e4de;">
    <p style="color: #44403c; font-size: 14px; line-height: 1.8; margin: 0;">
        TF-IDF with a linear SVM is a strong baseline. Trigram features capture phrases like "forest fire" and "building collapse" that are almost always literal. It breaks on metaphorical usage and sarcasm where the same words appear in completely different contexts.
    </p>
</div>
<div style="
    margin: 0 auto 12px;
    padding: 18px 24px;
    background: #faf9f6;
    border-radius: 10px;
    border: 1px solid #e8e4de;">
    <p style="color: #44403c; font-size: 14px; line-height: 1.8; margin: 0;">
        The BiLSTM adds sequence awareness but the gain over TF-IDF is modest. Tweets are short, around 15 words median so there is limited sequential structure to exploit. GloVe helps with vocabulary coverage but the fundamental limitation remains: one vector per word regardless of context.
    </p>
</div>
<div style="
    margin: 0 auto 12px;
    padding: 18px 24px;
    background: #faf9f6;
    border-radius: 10px;
    border: 1px solid #e8e4de;">
    <p style="color: #44403c; font-size: 14px; line-height: 1.8; margin: 0;">
        BERT wins because it builds contextual embeddings. The word "fire" gets a different representation in "the building is on fire" versus "he got fired" because self-attention considers the full sentence at once. A few epochs of fine-tuning was enough since the pre-trained weights already encode deep language understanding.
    </p>
</div>
<div style="
    margin: 0 auto 12px;
    padding: 18px 24px;
    background: #faf9f6;
    border-radius: 10px;
    border: 1px solid #e8e4de;">
    <p style="color: #44403c; font-size: 14px; line-height: 1.8; margin: 0;">
        The keyword column carries real signal. Keywords like "derailment" and "typhoon" have near-perfect disaster rates while others like "blazing" and "siren" are almost always metaphorical. A simple keyword check could handle the obvious cases before passing the rest to a full model.
    </p>
</div>
<div style="
    margin: 0 auto 12px;
    padding: 18px 24px;
    background: #faf9f6;
    border-radius: 10px;
    border: 1px solid #e8e4de;">
    <p style="color: #44403c; font-size: 14px; line-height: 1.8; margin: 0;">
        Tweets that all three models get wrong are often unclear by nature. Indirect references to emergencies, news headlines missing context and casual use of disaster vocabulary all contribute. These are cases where the text alone does not contain enough information to classify correctly.
    </p>
</div>